# PreProcessing: Deteksi Outlier & Imputasi Missing Value serta Ekstraksi Fitur Pada NO2

### 1. Deteksi Outliers pada Data Kualitas Udara di Kecamatan Kerek

In [1]:
import pandas as pd

# Membaca data gabungan kualitas udara Tuban
df = pd.read_csv("mystorage/KualitasUdaraKerek.csv")

# Daftar kolom fitur polutan yang akan dicek outlier-nya
# (Sesuaikan dengan nama kolom polutan yang ada di DataFrame Anda)
fitur_polutan = ['CO', 'NO2', 'SO2', 'O3']

for kolom in fitur_polutan:
    if kolom in df.columns:
        Q1 = df[kolom].quantile(0.25)
        Q3 = df[kolom].quantile(0.75)
        IQR = Q3 - Q1

        lower_bound = Q1 - (1.5 * IQR)
        upper_bound = Q3 + (1.5 * IQR)

        
        outliers_iqr = df[
            (df[kolom] < lower_bound) | 
            (df[kolom] > upper_bound)
        ]

        print(f"=== Parameter: {kolom} ===")
        print(f"Jumlah Outlier : {len(outliers_iqr)}")
        if len(outliers_iqr) > 0:
            print(outliers_iqr[['date', kolom]].head())
        print("-" * 35 + "\n")

FileNotFoundError: [Errno 2] No such file or directory: 'mystorage/KualitasUdaraKerek.csv'

### 2. Deteksi Outliers pada NO2 Untuk Ekstraksi Fitur

In [1]:
import pandas as pd

# Membaca data NO2
df = pd.read_csv("mystorage/NO2 Kerek.csv")

# Daftar kolom fitur polutan yang akan dicek outlier-nya
fitur_polutan = ['NO2']

for kolom in fitur_polutan:
    if kolom in df.columns:
        # Konversi kolom ke tipe data numerik (jika ada teks/string/anomali, akan diubah jadi NaN)
        df[kolom] = pd.to_numeric(df[kolom], errors='coerce')
        
        Q1 = df[kolom].quantile(0.25)
        Q3 = df[kolom].quantile(0.75)
        IQR = Q3 - Q1

        lower_bound = Q1 - (1.5 * IQR)
        upper_bound = Q3 + (1.5 * IQR)

        outliers_iqr = df[
            (df[kolom] < lower_bound) | 
            (df[kolom] > upper_bound)
        ]

        print(f"=== Parameter: {kolom} ===")
        print(f"Jumlah Outlier : {len(outliers_iqr)}")
        if len(outliers_iqr) > 0:
            print(outliers_iqr[['date', kolom]].head())
        print("-" * 35 + "\n")

=== Parameter: NO2 ===
Jumlah Outlier : 4
                         date       NO2
62   2026-08-10T00:00:00.000Z  0.000105
167  2026-03-29T00:00:00.000Z  0.000094
296  2025-09-23T00:00:00.000Z  0.000097
364  2026-05-27T00:00:00.000Z  0.000108
-----------------------------------



### 3. Merubah Outliers NO2 Menjadi NaN serta Imputasi Missing Values dan Outliers (NaN)

In [5]:
import pandas as pd
import numpy as np

# 1. Load data
df = pd.read_csv("mystorage/NO2 Kerek.csv")
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').set_index('date')

cols = ['NO2']

# 2. Ubah outliers pada data asli menjadi NaN terlebih dahulu
for col in cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Ganti nilai di luar batas IQR menjadi NaN
    df.loc[(df[col] < lower_bound) | (df[col] > upper_bound), col] = np.nan

# 3. Sekarang lakukan imputasi (missing value asli + outliers yang jadi NaN) sekaligus
df_clean = df.interpolate(method='time').ffill().bfill()

# 4. Cek hasil akhir
print("Sisa Missing Value:", df_clean.isnull().sum().sum())
df_clean.to_csv('NO2-Kerek-Clean.csv')

Sisa Missing Value: 0


### 4. Install Library TSFEL Untuk Ekstraksi Fitur

In [6]:
pip install tsfel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 81.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 58.7 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12/12 [tsfel]m11/12 [tsfel]odels]]
Note: you may need to restart the kernel to use updated packages.


In [7]:
pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [8]:
pip install numpy

Note: you may need to restart the kernel to use updated packages.


### 5. Ekstraksi Menjadi 68 Fitur 

In [9]:
import pandas as pd
import numpy as np
import inspect
import tsfel.feature_extraction.features as tsfel_features

# ---------- 1. Muat dan bersihkan data ----------
df = pd.read_csv('mystorage/NO2-Kerek-Clean.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

target_pollutant = 'NO2'

# --- FIX: paksa kolom target jadi numerik, nilai yang gagal dikonversi -> NaN ---
df[target_pollutant] = pd.to_numeric(df[target_pollutant], errors='coerce')

n_missing_before = df[target_pollutant].isna().sum()
print(f"Jumlah nilai non-numerik/kosong yang dikonversi jadi NaN: {n_missing_before}")

Q1 = df[target_pollutant].quantile(0.25)
Q3 = df[target_pollutant].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
df.loc[(df[target_pollutant] < lower_bound) | (df[target_pollutant] > upper_bound), target_pollutant] = np.nan

df_clean = df.set_index('date').interpolate(method='time').ffill().bfill()

fs = 1
signal_1d = df_clean[target_pollutant].astype(float).values

# ---------- 2. Daftar PERSIS fitur yang diminta ----------
FEATURE_LIST = """abs_energy auc autocorr average_power calc_centroid calc_max calc_mean
calc_median calc_min calc_std calc_var dfa distance ecdf ecdf_percentile ecdf_percentile_count
ecdf_slope entropy fundamental_frequency higuchi_fractal_dimension hist_mode human_range_energy
hurst_exponent interq_range kurtosis lempel_ziv lpcc max_frequency max_power_spectrum
maximum_fractal_length mean_abs_deviation mean_abs_diff mean_diff median_abs_deviation
median_abs_diff median_diff median_frequency mfcc mse negative_turning neighbourhood_peaks
petrosian_fractal_dimension pk_pk_distance positive_turning power_bandwidth rms skewness slope
spectral_centroid spectral_decrease spectral_distance spectral_entropy spectral_kurtosis
spectral_positive_turning spectral_roll_off spectral_roll_on spectral_skewness spectral_slope
spectral_spread spectral_variation spectrogram_mean_coeff sum_abs_diff wavelet_abs_mean
wavelet_energy wavelet_entropy wavelet_std wavelet_var zero_cross""".split()

print("Jumlah fitur yang diminta:", len(FEATURE_LIST))


def to_scalar(result):
    if isinstance(result, dict) and "values" in result:
        result = result["values"]
    if isinstance(result, (list, tuple, np.ndarray)):
        arr = np.asarray(result, dtype=float)
        return float(np.nanmean(arr))
    return float(result)


def extract_one(fn_name, signal, fs):
    fn = getattr(tsfel_features, fn_name)
    params = inspect.signature(fn).parameters
    if "fs" in params:
        result = fn(signal, fs)
    else:
        result = fn(signal)
    return to_scalar(result)


row = {}
for fn_name in FEATURE_LIST:
    row[fn_name] = extract_one(fn_name, signal_1d, fs)

extracted_features_final = pd.DataFrame([row])

print(f"Berhasil! Jumlah fitur yang dihasilkan untuk {target_pollutant}: {extracted_features_final.shape[1]}")
extracted_features_final.to_csv(f'{target_pollutant}-Kerek-TSFEL.csv', index=False)

/opt/conda/envs/esa-snap/lib/python3.12/site-packages/joblib/_multiprocessing_helpers.py:44: UserWarning: [Errno 2] No such file or directory.  joblib will operate in serial mode
  warnings.warn("%s.  joblib will operate in serial mode" % (e,))


Jumlah nilai non-numerik/kosong yang dikonversi jadi NaN: 0
Jumlah fitur yang diminta: 68
Berhasil! Jumlah fitur yang dihasilkan untuk NO2: 68


### 6. Mengelompokkan Fitur Hasil Ekstraksi Berdasarkan Domain

In [10]:
import pandas as pd

# ---------- 1. Muat hasil ekstraksi TSFEL ----------
df_features = pd.read_csv('mystorage/NO2-Kerek-TSFEL.csv')

# ---------- 2. Mapping resmi fitur -> domain (sesuai features.json TSFEL) ----------
FEATURE_DOMAIN_MAP = {
    # Statistical (21)
    "abs_energy": "statistical", "average_power": "statistical", "calc_max": "statistical",
    "calc_mean": "statistical", "calc_median": "statistical", "calc_min": "statistical",
    "calc_std": "statistical", "calc_var": "statistical", "ecdf": "statistical",
    "ecdf_percentile": "statistical", "ecdf_percentile_count": "statistical",
    "ecdf_slope": "statistical", "entropy": "statistical", "hist_mode": "statistical",
    "interq_range": "statistical", "kurtosis": "statistical", "mean_abs_deviation": "statistical",
    "median_abs_deviation": "statistical", "pk_pk_distance": "statistical", "rms": "statistical",
    "skewness": "statistical",

    # Temporal (15)
    "auc": "temporal", "autocorr": "temporal", "calc_centroid": "temporal",
    "distance": "temporal", "lempel_ziv": "temporal", "mean_abs_diff": "temporal",
    "mean_diff": "temporal", "median_abs_diff": "temporal", "median_diff": "temporal",
    "negative_turning": "temporal", "neighbourhood_peaks": "temporal",
    "positive_turning": "temporal", "slope": "temporal", "sum_abs_diff": "temporal",
    "zero_cross": "temporal",

    # Spectral (26)
    "fundamental_frequency": "spectral", "human_range_energy": "spectral", "lpcc": "spectral",
    "max_frequency": "spectral", "max_power_spectrum": "spectral", "median_frequency": "spectral",
    "mfcc": "spectral", "power_bandwidth": "spectral", "spectral_centroid": "spectral",
    "spectral_decrease": "spectral", "spectral_distance": "spectral", "spectral_entropy": "spectral",
    "spectral_kurtosis": "spectral", "spectral_positive_turning": "spectral",
    "spectral_roll_off": "spectral", "spectral_roll_on": "spectral", "spectral_skewness": "spectral",
    "spectral_slope": "spectral", "spectral_spread": "spectral", "spectral_variation": "spectral",
    "spectrogram_mean_coeff": "spectral", "wavelet_abs_mean": "spectral", "wavelet_energy": "spectral",
    "wavelet_entropy": "spectral", "wavelet_std": "spectral", "wavelet_var": "spectral",

    # Fractal (6) - domain terpisah di TSFEL
    "dfa": "fractal", "higuchi_fractal_dimension": "fractal", "hurst_exponent": "fractal",
    "maximum_fractal_length": "fractal", "mse": "fractal", "petrosian_fractal_dimension": "fractal",
}

# ---------- 3. Kelompokkan kolom sesuai domain ----------
domain_columns = {"statistical": [], "temporal": [], "spectral": [], "fractal": [], "unknown": []}
for col in df_features.columns:
    domain = FEATURE_DOMAIN_MAP.get(col, "unknown")
    domain_columns[domain].append(col)

for domain, cols in domain_columns.items():
    print(f"{domain.upper()} ({len(cols)}): {cols}")

# ---------- 4. (Opsional) Pisah jadi DataFrame terpisah per domain ----------
df_statistical = df_features[domain_columns["statistical"]]
df_temporal = df_features[domain_columns["temporal"]]
df_spectral = df_features[domain_columns["spectral"]]
df_fractal = df_features[domain_columns["fractal"]]

# ---------- 5. (Opsional) Simpan masing-masing sebagai CSV terpisah ----------
df_statistical.to_csv('NO2_fitur_statistical.csv', index=False)
df_temporal.to_csv('NO2_fitur_temporal.csv', index=False)
df_spectral.to_csv('NO2_fitur_spectral.csv', index=False)
df_fractal.to_csv('NO2_fitur_fractal.csv', index=False)

STATISTICAL (21): ['abs_energy', 'average_power', 'calc_max', 'calc_mean', 'calc_median', 'calc_min', 'calc_std', 'calc_var', 'ecdf', 'ecdf_percentile', 'ecdf_percentile_count', 'ecdf_slope', 'entropy', 'hist_mode', 'interq_range', 'kurtosis', 'mean_abs_deviation', 'median_abs_deviation', 'pk_pk_distance', 'rms', 'skewness']
TEMPORAL (15): ['auc', 'autocorr', 'calc_centroid', 'distance', 'lempel_ziv', 'mean_abs_diff', 'mean_diff', 'median_abs_diff', 'median_diff', 'negative_turning', 'neighbourhood_peaks', 'positive_turning', 'slope', 'sum_abs_diff', 'zero_cross']
SPECTRAL (26): ['fundamental_frequency', 'human_range_energy', 'lpcc', 'max_frequency', 'max_power_spectrum', 'median_frequency', 'mfcc', 'power_bandwidth', 'spectral_centroid', 'spectral_decrease', 'spectral_distance', 'spectral_entropy', 'spectral_kurtosis', 'spectral_positive_turning', 'spectral_roll_off', 'spectral_roll_on', 'spectral_skewness', 'spectral_slope', 'spectral_spread', 'spectral_variation', 'spectrogram_mean_